# 03 · Olay çalışması (event study): KAP haberleri fiyatı ne kadar oynatıyor?

papi'nin kalbi KAP olayları. Asıl soru: "Bu tip bir bildirim gelince hisse **piyasadan bağımsız olarak**
ne yapıyor?" Akademik yöntem: MacKinlay (1997), *Event Studies in Economics and Finance*.

1. **Tahmin penceresi** (olaydan önceki ~250 → 30 gün): piyasa modeli R_hisse = α + β·R_piyasa + ε'yi kur.
2. **Olay penceresi** (ör. −1 … +5 gün): **anormal getiri** AR = R_hisse − (α + β·R_piyasa).
3. **CAR** (kümülatif anormal getiri) = AR'ların toplamı. Aynı tip birçok olayın CAR'ını ortala.

**Veri ön koşulu:** papi'de KAP olayları şimdilik sadece 26.10.2026'dan beri var. Eve dönünce geçmişi yükle
(saatler sürer, KAP'a 1 istek/sn; `papi run` arka planda işler):

```
uv run papi kap backfill --from 2021-01-01 --to 2026-10-26 --tickers THYAO,TUPRS,CCOLA
```

Fonksiyonları sentetik veriyle şimdiden yazıp kontrol edebilirsin.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))  # learn/ klasöründen çalıştır
import kontrol as K
import papidata as pdx

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams["figure.figsize"] = (11, 4)

In [ ]:
def market_model(s: pd.Series, m: pd.Series, event: pd.Timestamp, est_window: tuple[int, int] = (-250, -30)) -> tuple[float, float]:
    """(alpha, beta): olay gününe göre göreli [est_window[0], est_window[1]) işlem günlerinde
    s = alpha + beta·m regresyonu. s, m: günlük getiriler (aynı index), event: s.index içinde bir gün.

    İpucu: i = s.index.get_loc(event); pencere s.iloc[i + a : i + b]. np.polyfit(x, y, 1) → [eğim, sabit].
    """
    # TODO
    raise NotImplementedError

In [ ]:
K.market_model(market_model)

In [ ]:
def car(s: pd.Series, m: pd.Series, event: pd.Timestamp, est_window: tuple[int, int] = (-250, -30),
        window: tuple[int, int] = (-1, 5)) -> pd.Series:
    """Kümülatif anormal getiri: index = göreli gün (window[0] … window[1], ikisi dahil),
    değer = o güne kadarki AR toplamı (window[0]'dan başlayarak)."""
    # TODO
    raise NotImplementedError

In [ ]:
K.car(car)

## Gerçek veri (KAP geçmişi yüklendikten sonra)

In [ ]:
px = pd.read_csv(pdx.DATA_DIR / "prices_adj.csv", index_col=0, parse_dates=True)
r = np.log(px).diff()

def event_day(ts: pd.Timestamp, index: pd.DatetimeIndex) -> pd.Timestamp | None:
    """Olayın fiyata yansıdığı ilk seans: 18:00'den sonra (seans kapanışı) duyurulduysa ertesi işlem günü."""
    d = ts.tz_localize(None).normalize()
    if ts.hour >= 18:
        d += pd.Timedelta(days=1)
    pos = index.searchsorted(d)
    return index[pos] if pos < len(index) else None

TICKER = "THYAO"
ev = pdx.events(TICKER)
print(len(ev), "olay"); ev["code"].value_counts().head(15)

In [ ]:
s, m = r[TICKER].dropna(), r["XU100"].dropna()
s, m = s.align(m, join="inner")
rows = []
for _, e in ev.iterrows():
    d = event_day(e["announced_at"], s.index)
    if d is None or s.index.get_loc(d) < 250 or s.index.get_loc(d) + 5 >= len(s):
        continue
    c = car(s, m, d)
    rows.append({"code": e["code"], "date": d, **{f"CAR[{k}]": v for k, v in c.items()}})
assert rows, "Yeterli geçmiş olay yok: önce KAP geçmişini yükle (yukarıdaki backfill komutu)"
cars = pd.DataFrame(rows)
by_code = cars.groupby("code").agg(n=("date", "size"), **{
    f"ort CAR[+{k}]": (f"CAR[{k}]", "mean") for k in (0, 1, 5)
}).sort_values("n", ascending=False)
by_code

In [ ]:
CODE = by_code.index[0]  # en sık olay tipi; başkasını da dene
sub = cars[cars["code"] == CODE].filter(like="CAR[")
mean, se = sub.mean(), sub.std() / np.sqrt(len(sub))
x = range(-1, 6)
plt.plot(x, mean.values, marker="o"); plt.fill_between(x, mean - 2 * se, mean + 2 * se, alpha=0.2)
plt.axhline(0, color="k", lw=0.5); plt.axvline(0, color="r", lw=0.5)
plt.title(f"{TICKER} · {CODE} · n={len(sub)} — ortalama CAR ± 2 standart hata");

## Yorumla

1. Hangi olay tipinde ortalama CAR ±2 standart hata bandının dışına çıkıyor (yani istatistiksel olarak anlamlı)?
2. Etki olay gününde mi (0) bitiyor, yoksa günlerce sürüyor mu? Sürüyorsa bu "piyasa verimsizliği" mi?
3. papi'nin önem (materiality) puanı yüksek olaylarda CAR gerçekten daha mı büyük? (`ev["materiality"]` ile grupla.)
4. Olay saatini (18:00 kuralı) yanlış alsan sonuç nasıl değişir?

## papi'ye taşıma (sonra)

Bu tam olarak PRD M3'teki "event-study engine". Olay tipi başına geçmiş CAR dağılımı, yeni bir KAP
bildirimi geldiğinde analiste ve Telegram uyarısına "bu tip haberlerde hisse geçmişte ortalama +%X
hareket etti" bilgisi olarak eklenebilir.